<a href="https://colab.research.google.com/github/akashde1998-Alpha/GCN-by-pytorch-/blob/main/GCN_via_pytorchgeometric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install Required Libraries
This cell installs `torch` and `torch_geometric`, which are essential libraries for building and working with Graph Neural Networks.

In [5]:
!pip install torch
!pip install torch_geometric



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.4 MB/s eta 0:00:00


### Import Necessary Modules
This cell imports various modules from `torch`, `torch.nn.functional`, and `torch_geometric` that will be used for defining the GCN model, handling data, and applying transformations.

In [6]:
import torch
import torch.nn.functional as F

import torch_geometric
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.datasets import Planetoid


### Load the Cora Dataset
This cell loads the Cora dataset, a commonly used benchmark for graph-based machine learning. `NormalizeFeatures()` is applied to normalize the node features, and the first graph in the dataset (`dataset[0]`) is assigned to the `data` variable.

In [7]:
dataset=torch_geometric.datasets.Planetoid(root='/Cora',name='Cora',transform=NormalizeFeatures())
data=dataset[0]

Processing...
Done!


### Inspect Dataset Properties
This cell prints various properties of the loaded Cora dataset, such as an example node's features, the total number of features per node, the number of classes, the total number of nodes, and the total number of edges. This helps in understanding the dataset's structure.

In [17]:
print(data)
print(data.x[36])
print(data.num_features)
print(dataset.num_classes)
print(data.num_nodes)
print(data.num_edges)


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
tensor([0.0455, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000])
1433
7
2708
10556


### GCN Model Definition and Instantiation
This cell defines a `GCN` model with three convolutional layers. The first layer maps input features to 16 hidden channels, the second layer maps 16 channels to 10 channels, and the third layer maps to the number of output classes. It includes ReLU activation and dropout after the first two layers. An instance of this `GCN` model is then created with `hidden_channels=16` (though the intermediate channels are hardcoded within the class definition), and its architecture is printed.

In [29]:
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
  def __init__(self, hidden_channels):
    super().__init__()
    torch.manual_seed(123456)
    self.conv1=GCNConv(data.num_features,16)
    self.conv2=GCNConv(16, 10)
    self.conv3=GCNConv(10, dataset.num_classes)
  def forward(self, x, edge_index ):
     x=self.conv1(x, edge_index)
     x=x.relu()
     # Dropout: randomly drops features during training.
     x=F.dropout(x,p=0.5, training=self.training)   # This line can be removed if we do not want to use dropout.
     x=self.conv2(x, edge_index)
     x=x.relu()
     x=F.dropout(x,p=0.5, training=self.training)
     x=self.conv3(x, edge_index)
     return x

print(model)

GCN(
  (conv1): GCNConv(1433, 16)
  (conv2): GCNConv(16, 10)
  (conv3): GCNConv(10, 7)
)
